# 06. VLA가 예측한 행동과 추론 시간 읽기

**목적:** 실제 가상 환경의 관측을 모델에 넣어 한 번 추론하고, 예측한 행동의 구성과 소요 시간을 해석합니다.

예상 시간 60분. [전체 안내](README.md).

![06. VLA가 예측한 행동과 추론 시간 읽기의 전체 과정](../교재그림/vla-inference-process.svg)

05에서는 모델에 전달할 입력 형식을 살펴봤습니다. 06은 준비된 SmolVLA 정책을 실제로 사용하여 **“이 장면에서 다음에 어떤 행동을 하겠는가?”** 를 확인합니다. 이때 행동은 숫자로 예측·저장되며 아직 환경에 적용하지 않습니다. 로봇이 물체를 옮기는 과정은 07에서 확인합니다.

### 전체 흐름과 실행 범위

환경 reset → 영상 2개·상태·과제 지시 → 모델 로드·전처리 → 한 번 추론 → 후처리 → 행동 파일 저장 → 배열·그래프·시간 해석 순서입니다.

`infer-01`은 04의 PNG나 05의 변수를 이어받지 않습니다. 자체 환경에서 얻은 초기 관측을 사용합니다. **기존에 같은 조건의 완료 결과가 있으면 다시 읽으므로 새 측정이 아닙니다.** 실제로 다시 측정하려면 `infer-02`처럼 쓰지 않은 이름을 사용합니다.

### 실행 셀과 출력 읽기

1. 설정 셀로 현재 커널의 ROOT·SESSION·RUNS와 실행 함수를 준비합니다.
2. 예제 셀 전체를 실행합니다. `run_lesson('infer', 'infer-01')`이 정상 종료한 뒤 행동 파일을 읽습니다. 파일 경로만 새 이름으로 바꾸면 파일이 생성되지 않습니다.
3. `actions.shape`와 앞 3시점의 값을 확인합니다. 유한한 값 검사 통과는 데이터 형식 확인이지 작업 성공이 아닙니다.
4. 그래프 셀을 실행하고, 아래 설명에 따라 축과 성분을 읽습니다.

| 출력 | 의미 | 오해하지 말아야 할 것 |
|---|---|---|
| `(1, 50, 7)` | 저장 묶음 1개·미래 시점 50개·행동 성분 7개 | 50초나 관측 50회가 아님 |
| `actions[0, :3]` | 첫 묶음의 앞 3시점 명령 | 실제로 이동한 로봇 위치가 아님 |
| 위 그래프의 3개 선 | 끝단 이동 명령 3개 | 절대 위치 궤적이 아님 |
| 아래 그래프의 4개 선 | 회전 명령 3개와 그리퍼 제어 1개 | 그리퍼 변화만으로 물체를 잡았다고 판단할 수 없음 |

**그래프 가로축은 0–49의 미래 행동 순번입니다. 초 단위 시간이 아닙니다.** 예측은 현재 관측에 근거한 것이므로 07에서 행동 후 새 관측을 얻으면 다음 묶음으로 교체됩니다. 50개의 명령이 모두 확정된 경로인 것은 아닙니다.

### 시간을 나누어 읽는 이유

| 필드 | 측정 범위 |
|---|---|
| `environment_setup_seconds` | 환경 준비와 reset |
| `model_load_seconds` | 정책과 처리 도구 로드 |
| `chunk_seconds` | 각 묶음의 추론·후처리 및 결과 전송 구간, 초 단위 목록 |

현재 runner는 입력 전처리 뒤에 묶음 타이머를 시작하고 GPU 완료를 기다립니다. 따라서 `chunk_seconds`를 전처리까지 포함한 전체 셀 실행 시간이라고 설명하지 않습니다. 06은 한 번 추론하므로 보통 길이 1 목록입니다. 다시 실행하면 프로세스가 모델을 새로 로드하므로 같은 모델을 메모리에 둔 연속 추론 성능 실험과도 구분합니다.

### TODO를 수행하는 정확한 순서와 상세 해설

실제 예제 결과가 준비된 뒤 기존 `notes` 셀에 아래 계산과 관찰 답안을 작성하세요. “1초”는 질문의 가정값이며 자신의 측정값은 `result`에서 읽습니다.

```python
# 50개 시점을 비교하여 행동 성분별 범위를 구합니다.
action_min = actions[0].min(axis=0).tolist()
action_max = actions[0].max(axis=0).tolist()
# 위 두 목록과 result['chunk_seconds']를 notes의 대응 항목에 기록합니다.
```

`actions[0]`은 `(50,7)`입니다. `axis=0`으로 줄이면 50행을 비교하여 열별 결과 7개가 남습니다. `axis=1`은 한 시점 안에서 이동·회전·그리퍼를 섞어 비교하므로 다른 질문에 답하게 됩니다. 각 min≤max인지, 두 목록의 길이가 7인지 확인합니다.

**정답의 핵심:** 한 번의 관측에서 미래 50개 행동을 계산했다는 뜻이지 카메라를 50번 다시 봤다는 뜻이 아닙니다. 저장된 값은 후처리된 결과이며 환경 전달 시 범위 제한 전이므로 min/max가 반드시 [-1,1] 안이라고 미리 적지 않습니다. 행동 배열이 정상이어도 실제 과제 성공은 07에서 검증해야 합니다.

재측정은 이전 실행이 끝난 뒤 새 이름으로 실행하고 그 결과 변수와 파일 경로를 함께 사용합니다. 기존 `actions`를 그대로 그려 놓고 새 측정 결과라고 해석하지 않습니다. 결과 파일 누락은 먼저 생성 실행의 오류를 확인하고, 완료 JSON만 남은 경우 기존 파일을 보존한 채 새 실행 이름으로 생성합니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'session-01'`은 같은 회차이며 결과는 `Data/logs/day5/session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.

## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

<!-- procedure:novnc-1744bd:start -->
![noVNC의 정상 대기와 완료](../교재그림/절차/novnc.svg)

03의 준비 검사와 05의 저장 배열 읽기는 새 실험 창을 열지 않습니다. 기존 완료 결과 재사용에도 새 창이 없습니다.
<!-- procedure:novnc-1744bd:end -->


JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

**이 노트북에서는 초기 장면과 모델 로드·추론 상태를 봅니다.** 행동을 예측·저장하지만 환경에 적용하지 않으므로 로봇은 움직이지 않습니다. 완료 창을 닫고 행동 배열·그래프를 확인합니다.

설정 셀의 `GUI`는 자동으로 켜집니다. 창 없이 시간 비교를 할 때는 `GUI = False`로 바꾸고 설정 셀을 다시 실행합니다. CLI 대체 실행은 `run_episode.py`에 `--gui`를 붙입니다. 노트북과 CLI를 동시에 실행하지 않습니다.

환경 초기화가 끝난 뒤 실제 초기 장면에서 창이 열립니다. 초기화 중 오류는 노트북 로그에서 확인합니다. 화면에는 실행 이름·모드·현재 상태·스텝·추론 횟수·적용 길이·직전 추론 초가 표시됩니다.

| 화면 상태 | 의미 |
|---|---|
| `INITIAL OBSERVATION` | 초기 관측 준비 완료 |
| `MODEL LOADING` | 모델 로드 중, 환경은 대기 |
| `INFERENCE` | 행동 계산 중, 환경은 대기 |
| `ACTION EXECUTION` | 예측 명령을 환경에 적용 중 |
| `FINISHED` | 종료 이유와 성공 여부 표시. 마지막 장면이며 더 움직이지 않음 |
| `INCOMPLETE / ERROR` | 미완료. 노트북의 원인 로그 확인 |

**정상 종료 화면을 관찰한 뒤 창을 닫으면 셀이 끝납니다.** 결과와 영상은 종료 화면 대기 전에 저장됩니다. 실행 도중 창을 닫으면 다음 안전한 처리 지점에서 사용자 중단으로 기록합니다. GPU 추론·모델 로딩 중에는 그 작업이 반환될 때까지 중단 반영이 늦을 수 있습니다. 창 조작은 표시용 복사본에만 적용되어 정책 카메라와 실제 실험 상태를 바꾸지 않습니다.

**기존 완료 결과를 읽으면 새 GUI는 열리지 않습니다.** 새 시연은 사용하지 않은 실행 이름으로 수행하고 변수·파일 경로도 함께 맞춥니다. GUI 사용 여부는 `result.json`의 `gui_enabled`에 남습니다. 화면 표시 비용이 있으므로 GUI 실행과 비GUI 실행의 시간을 같은 조건처럼 비교하지 않습니다.


**브라우저 탭을 닫아도 실험은 종료되지 않습니다.** 다시 같은 주소로 접속하면 진행 중 화면으로 돌아옵니다. FINISHED 후 화면 안의 MuJoCo 창을 닫아야 셀이 끝납니다. noVNC 연결과 탭은 다음 실험까지 유지하세요. 다른 학생 화면이 보이면 포트를 확인합니다.


## 슬라이드와 함께 진행하기

강사의 목적·흐름 설명을 들은 뒤 해당 셀을 실행하고, 출력과 TODO를 슬라이드의 확인 질문으로 해석합니다. 노트북 사전 정독을 전제하지 않습니다.

쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **예제·TODO·검증** → [5일차 슬라이드 38쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s034) · [배포 PDF 38쪽](../슬라이드/5일차.pdf#page=38)에서 42쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이전 자료의 위치와 필요한 입력 계약

[05_observation_action.ipynb의 「개념」](05_observation_action.ipynb)에 영상2·상태8·행동7이 있습니다. 다시 쓰면 raw RGB 영상은 각각 `(256,256,3)` uint8이며 방향 반전·CHW float 변환과 processor 정규화를 거칩니다. 상태는 위치3+axis-angle3+그리퍼 관절2, 행동은 OSC 끝단 이동3+회전3+그리퍼1이며 절대 좌표가 아닙니다.

이 노트북의 `run_lesson('infer','infer-01')`는 자체 환경 reset으로 초기 관측과 지시를 만들고 정책을 로드하므로 inspect-01 파일이나 05의 변수는 필요하지 않습니다. 필수 체크포인트·캐시는 시작 절에 있습니다. 출력 `Data/logs/day5/<SESSION>/infer-01/action-chunks.npy`는 `(1,50,7)`의 postprocessor 적용 후 행동 배열, result.json의 chunk_seconds는 각 추론 소요 초 목록입니다. 새 측정은 `infer-02` 같은 새 이름을 쓰며 기존 완료 결과를 읽는 것은 재측정이 아닙니다.

## 개념

정책을 strict=True로 로드합니다. load_vlm_weights=False는 전체 정책에 이미 포함된 backbone 가중치를 다시 다운로드하지 않는 설정이며 가중치를 생략하는 것이 아닙니다.

아래는 실제 환경 관측으로 한 번 추론하고 행동을 저장합니다. 아직 로봇 작업을 수행하지 않습니다. 모델 로드 시간과 행동 묶음 추론 시간을 나누어 읽으세요.

### 단계별 활동

진행: 모델 로드15분 → 실제 입력 추론15분 → 행동 묶음 시각화15분 → 지연·실패 해석15분. 출력의 각 열을 시간축에 대해 그려 변화를 확인합니다. 첫 묶음의 미래 행동은 다음 관측으로 다시 예측되므로 확정된 궤적이 아닙니다. infer-02로 재실행하고 초기화 시간과 추론 시간을 분리해 비교하세요.

In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA').is_dir())
SCRIPTS = ROOT / '10.VLA/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state, gui=GUI)


## 예제

<!-- procedure:lesson-42-e862f8:start -->
![예측하고 창을 닫은 뒤 읽기](../교재그림/절차/lesson-42.svg)

기존 완료 결과를 재사용하면 새 창이 없습니다. 새 실행에서는 FINISHED 후 MuJoCo 창을 닫습니다.
<!-- procedure:lesson-42-e862f8:end -->




### 실행 전 확인 — FINISHED 후 MuJoCo 창 닫기

**noVNC에서 `FINISHED`를 확인하면 화면 안의 MuJoCo 창 오른쪽 위 ×를 눌러 닫아주세요.** 창을 닫아야 아래 셀이 완료되고 다음 단계로 진행됩니다. 결과는 이미 저장되었지만, 마지막 장면을 관찰하도록 창을 유지하므로 노트북의 `[*]`와 경과 시간은 계속 표시됩니다. **브라우저 탭만 닫으면 셀은 끝나지 않습니다.** 실행 중 창을 닫으면 중단으로 처리되므로 `FINISHED`를 확인한 뒤 닫습니다.

06에서는 행동을 예측·저장만 하므로 로봇이 움직이지 않는 것이 정상입니다. 완료 창을 닫으면 행동 배열·추론 시간이 출력되고 다음 셀에서 그래프를 확인할 수 있습니다.

기존 완료 결과를 재사용하거나 `GUI=False`로 실행하면 새 GUI 창이 열리지 않으며, 창 닫기를 기다리지 않습니다.


In [ ]:
result = run_lesson('infer', 'infer-01')
import numpy as np
actions = np.load(RUNS/'infer-01/action-chunks.npy')
print('묶음수, 시점수, 행동차원:', actions.shape)
assert actions.shape == (1,50,7)
assert np.isfinite(actions).all()
print('앞3시점:', actions[0,:3])
print('추론 초:', result['chunk_seconds'])

### 입력과 실행 흐름 확인

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2,1,figsize=(10,5))
axes[0].plot(actions[0,:,:3]); axes[0].set_title('Relative translation action (normalized)')
axes[0].legend(['translation x', 'translation y', 'translation z'])
axes[1].plot(actions[0,:,3:]); axes[1].set_title('Rotation and gripper action')
axes[1].legend(['rotation x', 'rotation y', 'rotation z', 'gripper'])
for ax in axes:
    ax.set_xlabel('Future action index (0–49)')
    ax.set_ylabel('Postprocessed action value')
    ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## TODO

50시점 행동을 예측하는 데 1초가 걸렸다면 카메라를 초당50번 다시 보는 것인지 설명하세요. 행동 각 열의 최소·최대 값을 구하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**작성 위치와 출력:** 바로 아래 코드 셀의 `notes = {}`에 관찰 답안을 dict로 작성하고 노트북을 저장합니다. 새 정책 함수를 구현하는 TODO가 아닙니다. 코드 검토만 할 때는 아래 형식의 의미만 읽고, 실제로 관찰하지 않은 값은 `None`/`미측정`으로 남깁니다. notes 출력은 파일 저장이 아니므로 답안은 셀에 직접 적고 노트북 자체를 저장하세요.
**입력 자료:** actions 배열 (1,50,7), result[chunk_seconds], 행동 그래프.
**권장 답안 형식** (key 이름은 기록용 권장안이며 자동 채점 schema가 아닙니다):
```python
notes = {'action_min': None,
 'action_max': None,
 'chunk_seconds': None,
 'observations_vs_actions': '관측 한 번과 행동 묶음 50개 관계를 설명',
 'evidence_status': '미측정'}
```
**진행 힌트:** actions[0]을 고르면 (50,7)입니다. 시간축 axis=0으로 min/max를 구해 각각 길이7 목록으로 기록합니다. axis=1이면 시점별 값50개가 되므로 요구한 결과가 아닙니다. result["chunk_seconds"]는 목록입니다. 저장 배열은 postprocessor를 지난 값이고 환경에 보낼 때 -1–1로 clip하므로, 실측 최소·최대가 반드시 범위 안이라고 미리 쓰지 않습니다.
**강의 멘트:** “1초에 행동50개를 예측했다는 뜻이지 카메라를50번 새로 관측했다는 뜻이 아닙니다.”
**완료 확인:** 각 항목에 근거와 이유가 있고 실제 관찰/실측 값과 개념 설명을 구분했는지 확인합니다. 정답 문단의 수치는 관찰하지 않은 자기 실행 결과를 대신하지 않습니다.
<!-- todo-review:requirements:end -->

**이 TODO가 요구하는 계산:** `actions[0]`의 시간축 50개를 줄여 각 열의 min/max를 구하고 `.tolist()`로 길이7 Python 목록 두 개를 notes에 넣습니다. 열 순서는 이동 x/y/z, 회전3, 그리퍼1입니다. `len(action_min)==len(action_max)==7`, 모든 값 유한, 각 min≤대응 max를 확인합니다. chunk_seconds는 실제 result의 목록을 복사하고 “1초”는 질문의 가정값과 자기 실측을 구분합니다. 모델을 다시 측정하는 선택 활동은 `run_lesson('infer','infer-02')`를 앞 실행 종료 후 수행하고 해당 result의 model_load_seconds와 chunk_seconds를 각각 비교합니다.


In [ ]:
# 관찰 결과/실험 설정을 여기에 기록하세요.
notes = {}
notes

## 검증과 정답·복구

한 관측에서 미래50개 행동을 예측합니다. 재관측50회가 아닙니다. actions[0].min(axis=0), max(axis=0)로 각 행동의 범위를 구합니다. 값만으로 작업 성공을 판정하지 않습니다.

TODO 완료 예시(완료 결과가 있으면 같은 조건인지 확인하고 읽습니다):
```python
print(actions[0].min(axis=0), actions[0].max(axis=0))
```

파일·캐시·CUDA 오류는 [복구 안내](README.md#복구)와 이 노트북 시작 절에 있습니다. 필요한 복구는 전용 이미지·/opt/course 커널 확인, Data 마운트·배포 모델/캐시 존재 확인, 다른 GPU 작업 종료 후 한 프로세스 실행입니다. 누락 파일을 다운로드하거나 상태8을 잘라 우회하지 않습니다. CLI는 노트북과 같은 실험의 대체 실행입니다. 같은 조건의 완료 결과는 다시 읽습니다. 새 실험은 SESSION과 후속 노트북의 회차를 함께 바꾸세요. 실험 출력은 삭제하지 않습니다.

### 정답 해설 — 입력부터 결과까지

**행렬 읽기:** actions shape `(1,50,7)`에서 1은 저장 묶음 수, 50은 미래 행동 시점, 7은 행동 성분입니다. `actions[0].min(axis=0)`은 50개 시점을 줄여 각 성분의 최솟값 7개를 줍니다. axis=1이면 시점마다 성분을 섞어 다른 질문에 답합니다.

한 관측에서 50개를 예측했으므로 카메라를 50번 다시 본 것이 아닙니다. 모델 로드 시간과 chunk_seconds를 구분하고, 유한한 행렬이 나왔다는 것과 물체 조작 성공을 구분합니다.

**확인 질문:** 그리퍼 열이 바뀌는 시점을 보았다고 실제 물체를 잡았음을 확정할 수 있나요?

**적용·재확인:** 관찰 답안은 자신의 실제 파일·조건으로 작성합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 행동 묶음 [1,50,7]과 추론 지연 관찰

[5일차 슬라이드 38쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s034) · [배포 PDF 38쪽](../슬라이드/5일차.pdf#page=38)로 돌아가 **38–42쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

